# Notebook 14 — Bytes, encoding & binary-shaped files

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate → Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `13-regex-text-extraction.ipynb`

**Next up:** `15-subprocess-and-archives.ipynb`

---

## Learning objectives

- Treat encode/decode as explicit boundaries around APIs and disk.
- Decode safely when vendors emit malformed UTF-8.
- Peek binary headers without loading entire artifacts.

## Table of contents

1. **UTF-8 encode/decode**
2. **Error handlers (`errors=`)**
3. **`pathlib` bytes paths**
4. **Progressive drills — literals → sloppy bytes → sniff magic header**
5. **Exercise — stable SHA256 chunks reader**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · UTF-8 encode/decode

*Explanation:* HTTP bodies and protobuf-adjacent workflows hop between **`str`** and **`bytes`**.


In [ ]:
utf8 = "prompt €".encode()
print(utf8, utf8.decode())


## 2 · Error handlers (`errors=`)

*Explanation:* Offline scrapes often contain broken sequences — **`replace`** or **`ignore`** beats crashing ingestion.


In [ ]:
broken = b"hello \xff world"
print(broken.decode("utf-8", errors="replace"))


## 3 · `pathlib` byte slices

*Explanation:* Inspect tiny prefixes (magic numbers) before committing parsers.


In [ ]:
from pathlib import Path

tmp = Path("sample_blob.bin")
tmp.write_bytes(b"PK\x03\x04fake-zip-prefix")

try:
    header = tmp.read_bytes()[:4]
    print(header)
finally:
    tmp.unlink(missing_ok=True)


---

## Progressive drills — **easy → harder**

**Embedding caches** and **audio chunks** frequently arrive as bytes — boundary discipline prevents mysterious Unicode crashes.


### A · Easiest — hex fingerprints

Compare checksum prefixes quickly.


In [ ]:
blob = b"abc"
print(blob.hex())


### B · Medium — newline normalization bytes-side

Sometimes normalize `\r\n` before decoding.


In [ ]:
raw = b"line\r\nline\r\n"
text = raw.replace(b"\r\n", b"\n").decode()
print(text.splitlines())


### C · Harder — `memoryview` zero-copy slice

Large buffers benefit from slicing without copying immediately.


In [ ]:
payload = bytearray(b"HEADERPAYLOADTRAILER")
view = memoryview(payload)
chunk = view[6:13]
print(bytes(chunk))


### Exercise — chunked hex digest

Implement `sha256_hex_chunks(path: Path, chunk_size: int = 65536) -> str` returning lowercase hex SHA256 of **file contents** reading in chunks (use `hashlib.sha256`). Works for empty files.


In [ ]:
from hashlib import sha256
from pathlib import Path


def sha256_hex_chunks(path: Path, chunk_size: int = 65536) -> str:
    raise NotImplementedError


p = Path("_tmp_digest_demo.bin")
p.write_bytes(b"hello-world")
try:
    assert sha256_hex_chunks(p) == sha256(b"hello-world").hexdigest()
finally:
    p.unlink(missing_ok=True)
print("OK")


### Solution — sha256_hex_chunks

<details>
<summary>Click to expand</summary>

```python
from hashlib import sha256
from pathlib import Path

def sha256_hex_chunks(path: Path, chunk_size: int = 65536) -> str:
    h = sha256()
    with path.open("rb") as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()
```

</details>
